# Taller: monta tú un modelo de marketing mix

Segunda parte del seminario. En la primera hora has visto qué es una priori, qué es una
posteriori y por qué un modelo jerárquico presta información entre grupos. Aquí no vas a
repetir aquello con otros datos: vas a **construir un modelo desde cero**.

El recorrido:

1. Los datos.
2. El modelo que ajusta todo el mundo (y por qué miente).
3. Escalado.
4. Adstock y saturación: parámetros que no son betas.
5. Prioris. La parte larga.
6. Simular desde la priori, antes de mirar los datos.
7. Ajuste.
8. Diagnóstico.
9. Resultados.
10. La pregunta que le importa a alguien.

In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(RAIZ))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pymc as pm
import pytensor.tensor as pt
import arviz as az
import statsmodels.api as sm

from src.estilo import ACENTO, GRISES, aplicar_estilo
from src.mmm import (
    CANALES,
    CONTROLES,
    MAX_LAG,
    cargar_datos,
    escalar_controles,
    escalar_kpi,
    escalar_medios,
    hill,
    matriz_rezagos,
    pesos_adstock,
    vida_media,
)

aplicar_estilo()
SEMILLA = 42
rng = np.random.default_rng(SEMILLA)

## 1. Los datos

Un *marketing mix model* responde a una pregunta vieja: de todo lo que vendo, ¿cuánto se
debe a lo que me gasto en publicidad? Antes se hacía con una regresión y una hoja de
cálculo. Ahora se hace con una regresión y un modelo bayesiano, que es lo mismo pero
admitiendo en voz alta que los datos no dan para tanto.

Usamos los datos simulados de **Google Meridian**, la librería de MMM de Google. 156
semanas, cinco canales de pago con impresiones y gasto, dos controles y un KPI.

Meridian presume, con razón, de su modelo jerárquico por regiones: en su repositorio hay
un fichero por geografías (`geo_all_channels.csv`) donde cada región tiene su propio
efecto y todas se prestan información, exactamente lo que has visto con los condados de
Minnesota. **Aquí vamos a usar el modelo nacional.** No porque el jerárquico sobre, sino
porque hoy el tema no es la jerarquía: es todo lo demás que hay que decidir para que un
modelo bayesiano sea defendible.

In [ ]:
df = cargar_datos(RAIZ)
print(f"{len(df)} semanas, de {df['time'].min():%Y-%m-%d} a {df['time'].max():%Y-%m-%d}")
df.head()

In [ ]:
fig, ejes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

ejes[0].plot(df["time"], df["revenue"] / 1e6, color=ACENTO)
ejes[0].set_ylabel("Ingresos (M€)")

for canal, color in zip(CANALES, GRISES + [ACENTO]):
    ejes[1].plot(df["time"], df[f"{canal}_spend"] / 1e3, color=color, label=canal, lw=1)
ejes[1].set_ylabel("Inversión (k€)")
ejes[1].legend(ncol=5, fontsize=9)

fig.tight_layout()
plt.show()

### Ejercicio 1

Antes de modelar nada, mira los datos como si te los acabaran de pasar por correo:

- ¿Cuánto se ha invertido en total en cada canal y cuánto se ha ingresado?
- ¿Hay semanas sin inversión en algún canal?
- ¿Qué correlación hay entre la inversión de cada canal y los ingresos? ¿Y entre canales?

La última pregunta es la importante: si dos canales suben y bajan a la vez, ningún modelo
del mundo va a poder separarlos limpiamente. Ni bayesiano ni de los otros.

In [ ]:
# Tu código aquí

## 2. El modelo que ajusta todo el mundo

Ingresos contra inversión de cada canal, más los dos controles. Mínimos cuadrados. El
coeficiente de cada canal se lee directamente como **ROI**: euros de ingreso incremental
por euro invertido.

In [ ]:
X = df[[f"{c}_spend" for c in CANALES] + CONTROLES]
ols = sm.OLS(df["revenue"], sm.add_constant(X)).fit()

filas = [f"{c}_spend" for c in CANALES]
resumen = pd.DataFrame({
    "roi_ols": ols.params[filas].values,
    "ic_bajo": ols.conf_int().loc[filas, 0].values,
    "ic_alto": ols.conf_int().loc[filas, 1].values,
    "p_valor": ols.pvalues[filas].values,
    "corr_con_ingresos": [df[f].corr(df["revenue"]) for f in filas],
}, index=CANALES)

print(f"R² = {ols.rsquared:.3f}")
resumen.round(2)

Un $R^2$ de 0,24 y ningún canal significativo. Los intervalos de confianza van de perder
dinero a doblarlo, todos. Y si miras las correlaciones simples, tres de los cinco canales
correlacionan **negativamente** con la facturación.

Leído en frío, este modelo dice que la publicidad no sirve para nada. También podría estar
diciendo que la empresa invierte más justo cuando las cosas van peor. Con 156 semanas y
cinco canales que suben y bajan a la vez, las dos explicaciones caben igual de bien.

### Ejercicio 2

- Mira la matriz de correlaciones entre las inversiones de los canales. ¿Qué dos canales
  no vas a poder separar nunca?
- Quita un canal del modelo y vuelve a ajustar. ¿Cuánto se mueven los coeficientes de los
  demás?
- Si tuvieras que dar un número, ¿qué ROI le pondrías al canal 2? ¿Y con qué cara?

La salida fácil es pedir más datos. No los va a haber: la inversión la decide un equipo de
marketing, no un experimento aleatorizado, y lleva años decidiéndola igual. La otra salida
es escribir en el modelo lo que ya sabes del negocio. Eso es el resto del taller.

In [ ]:
# Tu código aquí

Ese modelo, además, asume dos cosas que son falsas:

1. Que el anuncio de esta semana no vende nada la semana que viene.
2. Que el euro un millón hace lo mismo que el primero.

Arreglar las dos tiene un precio: cada arreglo mete parámetros que los datos no
identifican solos. Y ahí es donde hay que decidir prioris.

## 3. Escalado

Los ingresos están en millones y las impresiones en decenas de millones. Si metes eso tal
cual en un modelo, cualquier priori que escribas es un disparate: `Normal(0, 5)` sobre una
variable que vale 8.000.000 no es una priori poco informativa, es una priori
absurdamente informativa a favor del cero.

Meridian escala así (y lo copiamos tal cual):

- **KPI**: se le resta la media y se divide por la desviación típica.
- **Medios**: cada canal se divide por la **mediana de sus semanas con inversión**. Una
  unidad = "una semana normal de ese canal".
- **Controles**: media cero, desviación típica uno.

In [ ]:
y = df["revenue"].to_numpy()
x = df[[f"{c}_impression" for c in CANALES]].to_numpy(float)
gasto = df[[f"{c}_spend" for c in CANALES]].to_numpy(float)
z = df[CONTROLES].to_numpy(float)

y_esc, y_media, y_sd = escalar_kpi(y)
x_esc, escala_medios = escalar_medios(x)
z_esc = escalar_controles(z)

gasto_total = gasto.sum(axis=0)

pd.DataFrame({
    "escala (mediana impresiones > 0)": escala_medios,
    "inversión total (€)": gasto_total,
    "semanas a cero": (x == 0).sum(axis=0),
}, index=CANALES).round(0)

### Ejercicio 3

- ¿Por qué la mediana de los valores positivos y no la media de todo? Fíjate en el canal
  que tiene semanas a cero.
- Comprueba que `y_esc` tiene media cero y desviación típica uno.
- ¿Qué valor tiene `x_esc` en una semana típica de cada canal? ¿Y en la mejor semana?

Este paso parece fontanería. No lo es: **todas las prioris del resto del taller están
escritas en esta escala**. Si cambias el escalado, cambias el modelo.

In [ ]:
# Tu código aquí

## 4. Adstock y saturación: parámetros que no son betas

Hasta ahora todo lo que has ajustado en tu vida tenía la forma "coeficiente por variable".
Un MMM no. Antes de multiplicar por nada, la inversión pasa por dos transformaciones con
parámetros propios, que también hay que estimar.

**Adstock** (el anuncio de hoy sigue vendiendo dentro de tres semanas):

$$\tilde{x}_{t} = \frac{\sum_{l=0}^{L} \alpha^{l}\, x_{t-l}}{\sum_{l=0}^{L} \alpha^{l}}, \qquad \alpha \in [0, 1]$$

Los pesos se normalizan a propósito: $\alpha$ **reparte** el efecto en el tiempo, no lo
infla. Meridian usa $L = 8$ semanas.

**Saturación de Hill** (el euro un millón rinde menos que el primero):

$$f(\tilde{x}) = \frac{\tilde{x}}{\tilde{x} + ec}$$

Va de 0 a 1 y vale exactamente 0,5 cuando $\tilde{x} = ec$: **$ec$ es el punto de media
saturación**, medido en medianas del canal. Meridian deja fijo el exponente (`slope = 1`),
que es lo que fuerza la curva a ser cóncava.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 3.6))

lags = np.arange(MAX_LAG + 1)
for alpha, color in zip([0.1, 0.4, 0.7, 0.9], GRISES[:3] + [ACENTO]):
    w = pesos_adstock(np.array([alpha]), MAX_LAG)[0]
    ejes[0].plot(lags, w, "o-", color=color,
                 label=f"α = {alpha} (vida media {vida_media(alpha):.1f} sem.)")
ejes[0].set_xlabel("semanas desde la impresión")
ejes[0].set_ylabel("peso")
ejes[0].legend(fontsize=9)

rejilla = np.linspace(0, 5, 200)
for ec, color in zip([0.3, 0.8, 2.0, 4.0], GRISES[:3] + [ACENTO]):
    ejes[1].plot(rejilla, hill(rejilla, ec), color=color, label=f"ec = {ec}")
ejes[1].set_xlabel("inversión (medianas del canal)")
ejes[1].set_ylabel("efecto relativo")
ejes[1].legend(fontsize=9)

fig.tight_layout()
plt.show()

### Ejercicio 4

- ¿Qué $\alpha$ hace falta para que el efecto de una impresión dure un mes? Usa
  `vida_media`.
- Con `ec = 0.8`, ¿qué porcentaje del efecto máximo consigue una semana normal
  ($\tilde{x} = 1$)? ¿Y una semana en la que inviertes el triple?
- Un canal de televisión y uno de búsqueda pagada, ¿tienen el mismo $\alpha$? ¿Y el mismo
  $ec$? Escribe tu respuesta antes de seguir: acabas de formular una priori.

In [ ]:
# Tu código aquí

## 5. Prioris

Ya tenemos el modelo entero salvo un detalle. Este es, escrito del todo:

$$
\begin{aligned}
y_t &\sim \mathcal{N}\!\left(\mu + \sum_m \beta_m f_{tm} + \sum_c \gamma_c z_{tc},\; \sigma^2\right) \\
f_{tm} &= \frac{\tilde{x}_{tm}}{\tilde{x}_{tm} + ec_m}, \qquad \tilde{x}_{tm} = \text{adstock}(x_{tm}; \alpha_m)
\end{aligned}
$$

Y estas son las prioris por defecto de Meridian, todas sobre datos ya escalados:

| Parámetro | Priori | Qué dice |
|---|---|---|
| $\mu$ | $\mathcal{N}(0, 5)$ | el nivel base, en desviaciones típicas del KPI |
| $\alpha_m$ | $\mathcal{U}(0, 1)$ | ni idea de cuánto dura el efecto |
| $ec_m$ | $\text{TruncNormal}(0{,}8;\ 0{,}8;\ [0{,}1;\ 10])$ | la media saturación cae cerca de una semana normal |
| $\gamma_c$ | $\mathcal{N}(0, 5)$ | los controles pueden hacer lo que quieran |
| $\sigma$ | $\text{HalfNormal}(5)$ | el ruido, en desviaciones típicas del KPI |

Falta $\beta_m$. Y aquí está el giro.

### $\beta$ no lleva priori

¿Qué priori le pondrías a $\beta_2$? Es el coeficiente de una curva de Hill sobre una
variable dividida por su mediana. Nadie tiene una intuición sobre eso. Nadie.

Sobre lo que sí tiene intuición el equipo de marketing es sobre el **ROI**: cuánto ingreso
incremental deja un euro invertido en ese canal. Así que la priori se pone ahí:

$$\text{roi}_m \sim \text{LogNormal}(0{,}2;\ 0{,}9)$$

y $\beta_m$ se **deduce**. El ingreso incremental del canal $m$ es lo que se pierde si lo
apagas del todo, y por construcción tiene que valer $\text{roi}_m$ por lo invertido:

$$\beta_m \cdot \sum_t f_{tm} \cdot sd(y) = \text{roi}_m \cdot \text{gasto}_m
\quad\Longrightarrow\quad
\beta_m = \frac{\text{roi}_m \cdot \text{gasto}_m}{sd(y) \cdot \sum_t f_{tm}}$$

$\beta_m$ deja de ser un parámetro con vida propia: es una cuenta que depende del ROI, del
adstock y de la saturación. Esto es lo mejor que hace Meridian, y no tiene nada que ver
con la programación probabilística: es reconocer que **la priori hay que escribirla en las
unidades en las que la gente sabe pensar**.

### Ejercicio 5

Traduce $\text{LogNormal}(0{,}2;\ 0{,}9)$ a algo que puedas decir en una reunión:

- ¿Cuál es el ROI mediano que asume esa priori?
- ¿Entre qué dos valores está el 90 % central?
- ¿Qué probabilidad le da a que un canal pierda dinero (ROI < 1)?

Pista: `scipy.stats.lognorm(s=0.9, scale=np.exp(0.2))`.

Y la pregunta incómoda: esa misma priori se le pone a los cinco canales. ¿Te parece
razonable, sabiendo que uno de ellos se lleva diez veces más presupuesto que otro?

In [ ]:
# Tu código aquí

## 6. Simular desde la priori

Una priori no se juzga mirando su fórmula, se juzga mirando **qué datos genera**. Vamos a
montar el modelo y a pedirle que invente 156 semanas de ingresos sin haber visto ni uno.

In [ ]:
rezagos = matriz_rezagos(x_esc, MAX_LAG)  # (semanas, rezagos, canales)

coords = {"canal": CANALES, "control": CONTROLES, "semana": df["time"].dt.date.astype(str)}

with pm.Model(coords=coords) as modelo:
    alpha = pm.Uniform("alpha", 0.0, 1.0, dims="canal")
    ec = pm.TruncatedNormal("ec", mu=0.8, sigma=0.8, lower=0.1, upper=10.0, dims="canal")
    roi = pm.LogNormal("roi", mu=0.2, sigma=0.9, dims="canal")
    gamma = pm.Normal("gamma", 0.0, 5.0, dims="control")
    mu = pm.Normal("mu", 0.0, 5.0)
    sigma = pm.HalfNormal("sigma", 5.0)

    # Adstock: media ponderada de los 9 rezagos, con pesos que suman uno.
    pesos = pesos_adstock(alpha, MAX_LAG)                 # (canales, rezagos)
    adstock = pt.sum(rezagos * pesos.T[None, :, :], axis=1)  # (semanas, canales)
    f = hill(adstock, ec)                                  # (semanas, canales)

    # Beta no es un parámetro libre: sale del ROI.
    beta = pm.Deterministic(
        "beta", roi * gasto_total / (y_sd * pt.sum(f, axis=0)), dims="canal"
    )

    contribucion = pm.Deterministic("contribucion", roi * gasto_total, dims="canal")

    media = mu + pt.dot(f, beta) + pt.dot(z_esc, gamma)
    pm.Normal("y", mu=media, sigma=sigma, observed=y_esc, dims="semana")

modelo

In [ ]:
with modelo:
    previa = pm.sample_prior_predictive(draws=500, random_seed=SEMILLA)

y_previo = previa.prior_predictive["y"].values.reshape(-1, len(df)) * y_sd + y_media

fig, ejes = plt.subplots(1, 2, figsize=(11, 3.6))

ejes[0].hist(y_previo.ravel() / 1e6, bins=80, color="#AAAAAA", density=True,
             label="simulado desde la priori")
ejes[0].hist(y / 1e6, bins=30, color=ACENTO, density=True, alpha=0.8, label="observado")
ejes[0].set_xlabel("Ingresos semanales (M€)")
ejes[0].legend(fontsize=9)

contrib_previa = previa.prior["contribucion"].values.reshape(-1, len(CANALES)).sum(axis=1)
ejes[1].hist(100 * contrib_previa / y.sum(), bins=80, color=ACENTO)
ejes[1].set_xlabel("% de los ingresos atribuido a los medios, según la priori")

fig.tight_layout()
plt.show()

### Ejercicio 6

El gráfico de la izquierda es el de siempre: ¿el modelo considera normales ingresos
imposibles? El de la derecha es el que de verdad importa.

- ¿Qué porcentaje de los ingresos le atribuye la priori a la publicidad, antes de ver un
  solo dato? Calcula su mediana y su percentil 95.
- Si tu jefe te dice que la publicidad explica como mucho el 30 % de la facturación,
  ¿qué priori sobre el ROI tendrías que escribir? Pruébala y vuelve a simular.
- ¿Qué pasa con la simulación si subes `sigma` a `HalfNormal(50)`?

Una priori que atribuye a la publicidad más ingresos de los que existen no es "poco
informativa". Es falsa. Y arreglarla ahora es gratis; después del ajuste, ya no.

In [ ]:
# Tu código aquí

## 7. Ajuste

Ahora sí, que vea los datos.

In [ ]:
RUTA_IDATA = RAIZ / "notebooks" / "idata" / "modelo_meridian.nc"

# Ponlo a False si el muestreo se te hace largo y prefieres cargar el resultado guardado.
MUESTREAR = True

if MUESTREAR:
    with modelo:
        idata = pm.sample(
            draws=1000, tune=1000, chains=4, target_accept=0.95, random_seed=SEMILLA
        )
        idata.extend(pm.sample_posterior_predictive(idata, random_seed=SEMILLA))
    RUTA_IDATA.parent.mkdir(parents=True, exist_ok=True)
    idata.to_netcdf(RUTA_IDATA, groups=["posterior", "sample_stats", "observed_data"])
else:
    idata = az.from_netcdf(RUTA_IDATA)
    with modelo:  # la predictiva posterior se recalcula, que es rápido
        idata.extend(pm.sample_posterior_predictive(idata, random_seed=SEMILLA))

## 8. Diagnóstico

Dos preguntas distintas que se confunden todo el rato:

- **¿Ha funcionado el muestreador?** `r_hat`, `ess_bulk`, `ess_tail`, divergencias.
- **¿Describe el modelo los datos?** La predictiva posterior.

Un modelo puede muestrear de maravilla y ser una tontería. Y al revés.

In [ ]:
resumen = az.summary(idata, var_names=["roi", "alpha", "ec", "gamma", "mu", "sigma"])
print("divergencias:", int(idata.sample_stats["diverging"].sum()))
print("r_hat máximo:", round(float(resumen["r_hat"].max()), 3))
print("ess_bulk mínimo:", int(resumen["ess_bulk"].min()))
resumen.round(2)

In [ ]:
az.plot_trace(idata, var_names=["roi", "sigma"], compact=True)
plt.tight_layout()
plt.show()

In [ ]:
az.plot_ppc(idata, num_pp_samples=100, colors=[ACENTO, "#7A7A7A", "black"])
plt.show()

### Ejercicio 7

- ¿Qué parámetro tiene el `ess` más bajo? ¿Te sorprende cuál es?
- Dibuja la posterior de `alpha` de cada canal contra su priori uniforme. ¿En cuáles ha
  aprendido algo el modelo y en cuáles te está devolviendo la priori con otro nombre?
- La predictiva posterior reproduce bien el centro de la distribución. Busca dónde falla.

Ojo con la conclusión fácil: que un parámetro no se mueva de su priori **no es un fallo
del muestreador**. Es información. Significa que estos datos no distinguen entre un canal
con memoria de una semana y uno con memoria de un mes, y que lo que salga por el otro lado
lo estás poniendo tú.

In [ ]:
# Tu código aquí

## 9. Resultados

El gráfico que resume el taller entero: priori contra posteriori del ROI de cada canal.

In [ ]:
roi_previo = previa.prior["roi"].values.reshape(-1, len(CANALES))
roi_post = idata.posterior["roi"].values.reshape(-1, len(CANALES))

fig, ejes = plt.subplots(1, len(CANALES), figsize=(13, 3), sharey=True)
bordes = np.linspace(0, 12, 61)
for i, (eje, canal) in enumerate(zip(ejes, CANALES)):
    eje.hist(roi_previo[:, i], bins=bordes, density=True, color="#CCCCCC", label="priori")
    eje.hist(roi_post[:, i], bins=bordes, density=True, color=ACENTO, alpha=0.85,
             label="posteriori")
    eje.set_title(canal, fontsize=11)
    eje.set_xlim(0, 12)
    eje.set_xlabel("ROI")
ejes[0].legend(fontsize=9)
fig.tight_layout()
plt.show()

In [ ]:
hdi = az.hdi(idata, var_names=["roi"], hdi_prob=0.9)["roi"].values
contrib = idata.posterior["contribucion"].values.reshape(-1, len(CANALES))

pd.DataFrame({
    "inversión (M€)": gasto_total / 1e6,
    "roi (mediana)": np.median(roi_post, axis=0),
    "roi hdi 90% bajo": hdi[:, 0],
    "roi hdi 90% alto": hdi[:, 1],
    "% ingresos (mediana)": 100 * np.median(contrib, axis=0) / y.sum(),
}, index=CANALES).round(2)

### Ejercicio 8

- Suma la contribución mediana de los cinco canales. ¿Qué porcentaje de la facturación
  atribuye el modelo a la publicidad? Compáralo con lo que decía la priori en el ejercicio
  6.
- Dibuja la **curva de respuesta** de un canal: multiplica su inversión por un factor
  entre 0 y 3, vuelve a pasar por adstock y Hill con los parámetros de la posteriori, y
  representa el ingreso incremental. Fíjate en dónde se dobla.
- ¿Coincide algún ROI con el que daba la regresión del ejercicio 2? ¿Cuál se ha movido más
  y por qué?

In [ ]:
# Tu código aquí

## 10. La pregunta que le importa a alguien

Nadie ha pedido nunca una posteriori. Piden a dónde va el dinero del trimestre que viene.

### Ejercicio 9

Con las muestras de la posteriori, calcula:

- La probabilidad de que el canal con mejor ROI mediano sea de verdad mejor que el
  segundo: `(roi_post[:, i] > roi_post[:, j]).mean()`.
- La probabilidad de que cada canal esté perdiendo dinero (ROI < 1).
- El intervalo del 90 % del ingreso incremental de un canal, en euros.

Y luego escribe **una frase**, sin la palabra "posteriori" dentro, que puedas decirle a
quien firma el presupuesto.

In [ ]:
# Tu código aquí

## Para llevarte a casa

- El modelo no ha descubierto los ROI: ha combinado unos datos flojos con una priori que
  has elegido tú. Cambia la priori y cambian los resultados. Eso no es un defecto del
  método bayesiano, es lo que el método te obliga a enseñar.
- La pregunta que hay que hacerle a cualquier MMM, propio o de proveedor, es la del
  ejercicio 6: **¿qué contribución de medios asume tu priori antes de ver los datos?** Si
  no saben responder, el número que te van a dar es el suyo, no el de los datos.
- Lo que dejamos fuera: el canal orgánico, la variable `Promo`, la estacionalidad con
  varios *knots*, la calibración del ROI con experimentos y —claro— el modelo jerárquico
  por regiones, que es donde de verdad brilla Meridian.